# 🚀 Fine-Tuning Snippy: The In-Browser JS Interaction Bot with Gemma 3 (270M)

This notebook demonstrates how to fine-tune **Gemma 3 270M** (`unsloth/gemma-3-270m-it`) as **Snippy** — an on-device JavaScript browser interaction bot. 

### Why Gemma 3 270M?
- **Ultra-Compact (~150MB INT8 bundle)**: Downloads and compiles in WebGPU in under a second.
- **Lightning-Fast On-Device Inference**: Instantaneous token generation directly in client browsers with LiteRT.js.
- **Ungated Mirror**: `unsloth/gemma-3-270m-it` runs out of the box without requiring HF tokens.

## Step 1: Install Dependencies
Use `%pip` magic (works in Colab & Jupyter) or `!uv pip install` if using an Astral `uv` environment.

In [24]:
# In Google Colab or standard Jupyter kernel:
%pip install -q torch transformers peft trl datasets litert-torch litert-lm

# If using Astral uv locally in terminal / notebook:
# !uv pip install -q torch transformers peft trl datasets litert-torch litert-lm

/Users/xprilion/.local/share/uv/tools/jupyterlab/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.


## Step 2: Fine-Tune Gemma 3 270M as 'Snippy' with LoRA / PEFT

We load `unsloth/gemma-3-270m-it` and fine-tune it on Snippy's browser-control instruction dataset.

In [25]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments
from peft import LoraConfig, get_peft_model, PeftModel
from trl import SFTTrainer, SFTConfig
from datasets import Dataset

# Using Gemma 3 270M Instruction-Tuned model!
MODEL_ID = "unsloth/gemma-3-270m-it"
OUTPUT_LORA_DIR = "./lora_adapter"
OUTPUT_MERGED_DIR = "./fine_tuned_gemma_merged"

# 1. Load Tokenizer & Base Model
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

# 2. Configure LoRA for Gemma 3
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
peft_model = get_peft_model(model, peft_config)

# 3. Snippy In-Browser JS Interaction Dataset
sample_data = [
    {
        "text": (
            "<start_of_turn>user\n"
            "Hi, who are you?<end_of_turn>\n"
            "<start_of_turn>model\n"
            "I am Snippy, your in-browser JavaScript interaction bot! Powered by Gemma 3 270M, LiteRT.js, and WebGPU, I can execute actions directly on this webpage.\n"
            "```js\nSnippy.showAlert('Snippy (Gemma 3 270M) is ready to control your browser!');\n```<end_of_turn>"
        )
    },
    {
        "text": (
            "<start_of_turn>user\n"
            "Snippy, change the background color to dark purple.<end_of_turn>\n"
            "<start_of_turn>model\n"
            "Changing background color to dark purple!\n"
            "```js\nSnippy.setBgColor('#1e1b4b');\nSnippy.showAlert('Background updated to Dark Purple 🌙');\n```<end_of_turn>"
        )
    },
    {
        "text": (
            "<start_of_turn>user\n"
            "Snippy, create a button called Explore Edge AI.<end_of_turn>\n"
            "<start_of_turn>model\n"
            "Here is your interactive button!\n"
            "```js\nSnippy.createButton('Explore Edge AI', 'Explore Edge AI button clicked!');\n```<end_of_turn>"
        )
    },
    {
        "text": (
            "<start_of_turn>user\n"
            "Snippy, celebrate with confetti!<end_of_turn>\n"
            "<start_of_turn>model\n"
            "🎉 Celebrating on-device WebGPU AI with confetti!\n"
            "```js\nSnippy.triggerConfetti();\nSnippy.showAlert('🎉 Gemma 3 270M WebGPU AI in action!');\n```<end_of_turn>"
        )
    },
    {
        "text": (
            "<start_of_turn>user\n"
            "Snippy, add a card about LiteRT.js.<end_of_turn>\n"
            "<start_of_turn>model\n"
            "Adding a card about LiteRT.js to the page layout.\n"
            "```js\nSnippy.createCard('LiteRT.js WebGPU', 'Google LiteRT brings high-performance Gemma 3 models straight to client browsers with zero server latency.');\n```<end_of_turn>"
        )
    }
]
dataset = Dataset.from_list(sample_data)

# 4. Training with SFTTrainer
sft_config = SFTConfig(
    dataset_text_field="text",
    max_length=256,
    output_dir="./results",
    num_train_epochs=5,
    per_device_train_batch_size=2,
    logging_steps=1,
    loss_type="nll"
)
trainer = SFTTrainer(
    model=peft_model,
    train_dataset=dataset,
    args=sft_config,
)
trainer.train()

# Save LoRA Adapter
peft_model.save_pretrained(OUTPUT_LORA_DIR)
tokenizer.save_pretrained(OUTPUT_LORA_DIR)
print("✅ Snippy Gemma 3 270M LoRA Adapter Saved!")

Truncating train dataset: 100%|██████████| 5/5 [00:00<00:00, 3568.41 examples/s]
Dropping fully masked examples from train dataset: 100%|██████████| 5/5 [00:00<00:00, 5430.22 examples/s]
/Users/xprilion/.local/share/uv/tools/jupyterlab/lib/python3.13/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
1,6.547552
2,7.127209
3,6.231329
4,6.618102
5,6.341316
6,6.853746
7,6.212180
8,6.338592
9,6.637869
10,6.421654


✅ Snippy Gemma 3 270M LoRA Adapter Saved!


## Step 3: Merge LoRA Adapter into Gemma 3 270M Base Model
Unload and merge Snippy's adapter weights into base weights.

In [26]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

MODEL_ID = "unsloth/gemma-3-270m-it"
OUTPUT_LORA_DIR = "./lora_adapter"
OUTPUT_MERGED_DIR = "./fine_tuned_gemma_merged"

# Load base model & adapter, then merge
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
base_model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float32, device_map="cpu")
peft_model = PeftModel.from_pretrained(base_model, OUTPUT_LORA_DIR)
merged_model = peft_model.merge_and_unload()

# Save Merged Checkpoint
merged_model.save_pretrained(OUTPUT_MERGED_DIR)
tokenizer.save_pretrained(OUTPUT_MERGED_DIR)
print("✅ Merged Snippy Gemma 3 270M Model Saved to:", OUTPUT_MERGED_DIR)

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.42it/s]

✅ Merged Snippy Gemma 3 270M Model Saved to: ./fine_tuned_gemma_merged


## Step 4: Convert Model to LiteRT Bundle (`.litertlm`)

Export model to LiteRT bundle format using `litert-torch export_hf` with `-b True`.

In [27]:
!litert-torch export_hf \
  ./fine_tuned_gemma_merged \
  ./litert_output \
  -b True \
  -q dynamic_int8

W0808 00:37:30.527000 8631 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
W0808 00:37:30.541000 8631 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W0808 00:37:31.472000 8631 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
============== Export Configuration ==============
aot_backend            : None
aot_compilation_config_dict : None
aot_soc_model          : None
assistant_model        : None
auto_model_override    : None
batch_size             : 1
bundle_litert_lm       : True
cache_implementation   : 'LiteRTLMCache'


## Step 5: Test Snippy with `litert-lm` Native Engine
Verify Snippy's generated JavaScript actions native response.

In [28]:
import litert_lm

# Initialize Engine with .litertlm bundle
engine = litert_lm.Engine('./litert_output/model.litertlm')
conv = engine.create_conversation()

res = conv.send_message('Snippy, change background color to dark purple.')
print("Snippy Response:", res['content'][0]['text'])

W0000 00:00:1786129689.414699  258656 litert_lm_loader.cc:291] Section not found: 
W0000 00:00:1786129689.415032  221652 litert_lm_loader.h:158] TFLite model type: TF_LITE_VISION_ENCODER not found for backend constraints. Skipping.
W0000 00:00:1786129689.415064  221652 litert_lm_loader.h:158] TFLite model type: TF_LITE_AUDIO_ENCODER_HW not found for backend constraints. Skipping.
W0000 00:00:1786129689.415069  221652 litert_lm_loader.h:174] TFLite model type: TF_LITE_VISION_ENCODER not found for prefer activation type. Use system's default backend activation type. System's default activation type for Text decoder is fp16. Vision encoder and audio encoder default is fp32.
W0000 00:00:1786129689.415072  221652 litert_lm_loader.h:174] TFLite model type: TF_LITE_AUDIO_ENCODER_HW not found for prefer activation type. Use system's default backend activation type. System's default activation type for Text decoder is fp16. Vision encoder and audio encoder default is fp32.
INFO: [environment.c

Snippy Response: Okay, I'm ready to help you with any background color changes you need! Please tell me what you want to change.



## Step 6: Deploy to Web Browser via LiteRT.js

Copy the exported model bundle and extract the TFLite FlatBuffer to the web server's static assets directory.

In [29]:
# Copy Converted Model Files to Local Web Demo Directory
import os
import shutil

SOURCE_MODEL = "./litert_output/model.litertlm"
DEST_DIR = "./web/public/models"

os.makedirs(DEST_DIR, exist_ok=True)
if os.path.exists(SOURCE_MODEL):
    # 1. Copy .litertlm bundle for Python backend engine
    shutil.copy(SOURCE_MODEL, os.path.join(DEST_DIR, "model.litertlm"))

    # 2. Extract TFLite FlatBuffer (starts with TFL3 magic) for WebGPU browser runtime
    with open(SOURCE_MODEL, "rb") as f:
        data = f.read()
    pos = data.find(b"TFL3")
    if pos != -1:
        tflite_data = data[pos - 4 :]
        with open(os.path.join(DEST_DIR, "model.tflite"), "wb") as f:
            f.write(tflite_data)
        print("✅ Extracted TFLite FlatBuffer and copied model files to web/public/models/")
    else:
        shutil.copy(SOURCE_MODEL, os.path.join(DEST_DIR, "model.tflite"))
        print("✅ Copied model files to web/public/models/")
else:
    print(f"⚠️ Source model not found at {SOURCE_MODEL}. Make sure Step 4 export finished.")

✅ Extracted TFLite FlatBuffer and copied model files to web/public/models/
